# Cleaning 2025 survey data

In this notebook, survey data collected during the 2025 iGEM Jamboree is cleaned. Two surveys have been conducted during the Jamboree: *iGEM ties* and *iGEM experience*. The same *iGEM ties* survey had been conducted in 2022 and 2023 as well, with added question about team member's academic field this year. The survey consists of three parts: demographics, relations, and tasks (analyzed before). This survey is done only with French teams in 2025, and the original `df_ties.csv` file contains French team members' answers to questions from all three parts, as well as their answers for the *iGEM experience* survey. Each row represents a person's answer to a certain question. The goal of this Jamboree data collection was to survey as many French team members to get the most possible team coverage.

On the other hand, the  *iGEM experience* survey was conducted with all iGEM team members and the goal was to get the most amount of responses, regardless of which team the member belong to, and its raw data can be found in `df_experience.csv`. This survey included 5 open-ended questions related to: Connection and belonging, Reflection on team collaboration, Beyond iGEM ideas, Supporting and limiting conditions, Phrases that capture what iGEM means to the team member.

Therefore, the goal of this notebook is to clean and organize the data files on the location `data/igem_ties_surveys/2025/survey_original_raw_data`, including cleaning fields that contain dictionaries of answers, and splitting the `df_ties.csv` into `survey_demographics`, `survey_relations`, `survey_tasks`, and `igem_experience_survey` CSV files. The iGEM experience French team answers in `df_ties.csv` files are also merged with the ones in `df_experience.csv` in this notebook.

In [1]:
import pandas as pd
import json, re
import hashlib

## 1. Combining survey tasks with attributions

One issue is that we noticed while trying to merge attribution and survey data (subtitle 8), is that the roster names that were used in the survey were fetched a few weeks before the Jamboree, and the names have since changed (some changed names and some API call issues). The attributions were fetched after the Jamboree, and there are around 90 names that have been slightly changed during this period. Therefore, the old roster file and new one are not completely the same, so some survey responses cannot be matched with new names in the attributions file. To solve this problem, we will:

1. First compare the new roster file and new attributions file to check if there were some API issues: does every (FullName, UUID) match?

2. Merge the new roster with the old one: add a new column to the old roster with the updated roster names, matching names by UUID. Check manually if everything makes sense.

3. In the survey, replace old participant names with the new ones from the "NewFullName" column from the rosters dataframe. 

4. Now, attributions and survey can be matched completely because they both have updated names (this is done in part 8)

In [2]:
# 2025 attributions dataframe
attributions_2025 = pd.read_csv('../results/attributions/results_attributions_2025.tsv', sep='\t')

In [3]:
# To check if the names in attributions and roster (what is used in the surveys) match correctly

# Team roster collected BEFORE the Jamboree - used in the surveys

attribution_names = list(attributions_2025["FullName"].unique())

team_rosters_2025_1 = pd.read_csv('../data/igem_scrapping_2025/team_roster_2025.tsv', sep='\t')
roster_names_1 = team_rosters_2025_1["FullName"].to_list()

not_found_names_1 = []
for name in attribution_names:
    if name not in roster_names_1:
        not_found_names_1.append(name)
print(len(not_found_names_1), not_found_names_1)

91 ['Yunfan Hu', 'Xin Guan', 'Ondra', 'Ching HUNG', 'Lasse Wrang Meyer', 'Stefana Dukic', 'Evan Seidler', 'Seoyeon Park', 'Jaehyun', 'Jaykeun', 'Mael Aernouts', 'Ana Radovic', 'Yazid HOBLOS', 'Daniela C', 'Michael Spädt', 'Jun Kang', 'Jui-Yu', 'YI-CHEN, CHIU', 'Cynthia Chuang', 'Chow Lok Man', 'Chan Lai Yin', 'LI HOULIN', 'Sabine Waigel', 'zheng huixin', 'Binbin He', 'XiangGao', 'Evangelos M. Masis', 'Rycki', 'Aleksandra Okrasa', 'Sacha Levy', 'Lily rose Charnley ', 'Jehlysa Albert', 'Julien Mercy', 'Latifa Rezzouk ', 'Nanami Shibata', 'Serena Chen', 'Ethan Huang', 'Shemile Poblete', 'Aiden Chee', 'Jooyoung Sim', 'Lou-Anne Bénéton', 'Yuyan Yang', 'Chenkai Luo', 'Long Pu', 'Yu Zheng', 'Mao Liao', "Jes O'Kelley", 'Vladislav Sladzevsky', 'Arnab Choudhary', 'Antreas Ermogenous', 'Dr. Anup A. Kale ', 'SHI ENXI', 'Leremos H', 'Andrea Londono ', 'Via Iseppon', 'Paula Veeramasu', 'Sebastian Mayer', 'Javier Figueroa', 'Ramazan Assan', 'Tanirbergen ', '唐勇强', 'Xuanwen jin', 'Xie Yutong', 'HSIA, H

In [4]:
# To check full rows where they don't match - unique names only

not_found_df = attributions_2025[attributions_2025["FullName"].isin(not_found_names_1)]

not_found_unique_df = not_found_df.drop_duplicates(subset=["FullName"])

not_found_unique_df

,FullName,Username,UUID,TeamID,Team,Year,Role,Task,TaskDescription
2484,Yunfan Hu,huyunfan,91b22f82-d912-4dc5-9ae5-51d91f2c8f4b,5984,BIT-LLM,2025,Student Leader,Analysis,NaN
3169,Xin Guan,guanxin,1c2a8c07-1661-455c-bdae-cff85e663460,6029,BSDFZ-CHINA,2025,Student,Background Research,researched papers and regulations related to t...
5614,Ondra,ondrejsvanda,54ad0894-448e-4154-92c5-635a82a654ae,5642,Brno Czech Republic,2025,Student,Investigation,NaN
6128,Ching HUNG,hungching1982,19b2be11-023e-44c0-9dd7-fb028d562380,5854,CCU-Taiwan,2025,Primary PI,Other,Dr. Hung played a crucial mentoring role in bu...
8706,Lasse Wrang Meyer,lassem,66988d2f-8c68-40bc-a466-8f1de0244111,5568,DTU-Denmark,2025,Secondary PI,Background Research,NaN
...,...,...,...,...,...,...,...,...,...
47641,Eva Schuiteman,eschuiteman,df14796f-02a7-4178-a420-47e5d81b6b73,5795,Utrecht,2025,Student,Analysis,NaN
48903,Wojciech Garstka,wojtek,98c621e8-1267-4f02-9d0e-fa9ddb25161d,5865,Warsaw,2025,Secondary PI,Analysis,NaN
49493,Jay Yasuda,aspenf,42dd33e1-b1e9-43e8-aba7-106596754b72,5934,Westview-SanDiego,2025,Student,Investigation,Carried out laboratory experiments such as bac...
50351,Juanli Cheng,juanlicheng,7bbb0cb6-96da-4eef-8879-af345b11017e,5853,YAU-China,2025,Secondary PI,Conceptualization,NaN


In [5]:
# To check if the names in attributions and roster match correctly

# Team roster collected AFTER the Jamboree - matches completely with attributions

team_rosters_2025_2 = pd.read_csv('../data/igem_scrapping_2025/team_roster_2025_after_jamboree.tsv', sep='\t')
roster_names_2 = team_rosters_2025_2["FullName"].to_list()

not_found_names_2 = []
for name in attribution_names:
    if name not in roster_names_2:
        not_found_names_2.append(name)
print(len(not_found_names_2), not_found_names_2)

0 []


### 1.1 Comparing new roster with new attributions

In [6]:
# Checking if each (FullName, Username, UUID) matches between attributions and rosters dataframes (both collected after 2025)

not_matched_persons_from_attributions = (
    attributions_2025.merge(
        team_rosters_2025_2
            .rename(columns={"RosterUUID": "UUID"})[["FullName","Username","UUID"]],
        how="left",
        on=["FullName","Username","UUID"],
        indicator=True,
        validate="many_to_one"  # optional: each (FullName,Username,UUID) should map to one roster row
    )
    .query('_merge == "left_only"')
    .drop(columns="_merge")
    [["FullName","Username","UUID", "Team"]]
)


not_matched_persons_from_rosters = (
    team_rosters_2025_2
    .rename(columns={"RosterUUID": "UUID"})
    .merge(
        attributions_2025[["FullName", "Username", "UUID"]],
        how="left",
        on=["FullName", "Username", "UUID"],
        indicator=True,
        validate="one_to_many"  # each roster row can appear many times in attributions
    )
    .query('_merge == "left_only"')
    .drop(columns=["_merge"])
    [["FullName","Username","UUID", "Team", "Role"]]
)

not_matched_persons_from_rosters_students = not_matched_persons_from_rosters[
    not_matched_persons_from_rosters["Role"].isin(["Student Leader", "Student Member"])
]

print("Not matched people from attributions:", len(not_matched_persons_from_attributions), '\n', not_matched_persons_from_attributions)
print('\n')
print("Not matched people from rosters:", len(not_matched_persons_from_rosters_students), '\n', not_matched_persons_from_rosters_students)


Not matched people from attributions: 0 
 Empty DataFrame
Columns: [FullName, Username, UUID, Team]
Index: []


Not matched people from rosters: 105 
                   FullName       Username  \
127           Altan Selçuk    altanselcuk   
128                   Dila        dilanur   
129              Ecem Aksu         ecem-a   
130    Ece Sude ASLANDOĞAN          esa13   
131          Elif Kalyoncu  elifkalyoncuu   
...                    ...            ...   
42407           ZHANG ENPU          zep23   
45281               Arya M      aryamanda   
45282             Debanshi          djain   
45301              Laxmini       luxpatel   
45326            Victoria    vdalzellvill   

                                       UUID           Team            Role  
127    9af23794-9cb0-403f-9f13-1a92b938c937          AEGIS  Student Leader  
128    fee7550e-97a2-4441-9963-b0a2693b706a          AEGIS  Student Member  
129    97a8a360-7093-4305-9058-b50d07f71142          AEGIS  Student Member  


There seems to be some issue with API calls: running the code for fetching attributions or rosters can lead to slightly different results each time (some names get mixed up). This mistake should be looked into. 
Normally, there are no (FullName, Username, UUID) tuples that exist in attributions but don't exist in the roster dataframe, But, there are students who are present in the roster but not in the attributions.

### 1.2 Merge new roster with old one

In [7]:
old_roster = team_rosters_2025_1.copy()
new_roster = team_rosters_2025_2.copy()

name_map = (
    new_roster.loc[:, ["RosterUUID", "FullName"]]
    .dropna(subset=["RosterUUID"])
    .drop_duplicates(subset=["RosterUUID"], keep="last")   # keep last if duplicates exist
    .rename(columns={"FullName": "NewFullName"})
)

# Left-merge onto the OLD roster
old_roster_with_new_names = old_roster.merge(name_map, on="RosterUUID", how="left")

# Number of different names between the old and new roster
if "FullName" in old_roster_with_new_names.columns:
    changed = (
        old_roster_with_new_names["NewFullName"].notna()
        & (old_roster_with_new_names["NewFullName"] != old_roster_with_new_names["FullName"])
    )
    print("Rows with a different name in the new roster:",
          int(changed.sum()), "of", len(old_roster_with_new_names))


# old_roster_with_new_names.to_csv("../data/igem_scrapping_2025/team_rosters_2025_1_with_new_names.tsv", sep="\t", index=False)

old_roster_with_new_names

Rows with a different name in the new roster: 98 of 10186


,Year,TeamID,Team,Username,FullName,Role,RosterUUID,NewFullName
0,2025,5794,ABOA,paulikallio,Pauli Kallio,Primary PI,da424895-3680-4cf4-83a3-3183eba72227,Pauli Kallio
1,2025,5794,ABOA,kalisa,Kalisa Volfovich,Student Leader,4c4d9945-3245-4c0c-ad59-9c8a9aa39add,Kalisa Volfovich
2,2025,5794,ABOA,riinamatas,Riina Mätäsniemi,Student Leader,d67bddc5-ef7d-4b13-aa03-815dc4af6e49,Riina Mätäsniemi
3,2025,5794,ABOA,aisa-khormali,Aisa,Student Member,49c92ea3-60fe-422e-a702-99b1d58bbda8,Aisa
4,2025,5794,ABOA,amandapynnonen,Amanda Pynnönen,Student Member,7e13fcb8-a467-4d7b-b7bc-bdd2dbfbdcc2,Amanda Pynnönen
...,...,...,...,...,...,...,...,...
10181,2025,5822,ZZU-China,michellejxr,Xiaoran Ji,Student Member,5e77d51a-f2a8-4909-a46b-48ec2e7ef0a4,Xiaoran Ji
10182,2025,5822,ZZU-China,yufei1228,Yufei Huang,Student Member,6dd928c7-6f2b-4ede-bc84-17d406aead33,Yufei Huang
10183,2025,5822,ZZU-China,charline,Yutong Liu,Student Member,061957f3-2338-40fd-966a-de0f78d03ccb,Yutong Liu
10184,2025,5822,ZZU-China,xyx-zzu,Yuxuan Xia,Student Member,5bd794e3-62d9-412e-9a83-4e132c67323c,Yuxuan Xia


### 1.3 Replace old survey names with new ones

In [8]:
# Survey Dataframes: ties and experience 2025 surveys (raw data) 

df_ties = pd.read_csv('../data/igem_ties_surveys/2025/survey_original_raw_data/df_ties.csv')
df_experience = pd.read_csv('../data/igem_ties_surveys/2025/survey_original_raw_data/df_experience.csv')

In [9]:
# Mapping old with new names
name_map_df = (
    old_roster_with_new_names.loc[old_roster_with_new_names["NewFullName"].notna(), ["FullName", "NewFullName"]]
    .drop_duplicates(subset=["FullName"], keep="last")
)
name_map = dict(zip(name_map_df["FullName"], name_map_df["NewFullName"]))

# Function that returns a new datframe that has replaced old participant names with new ones
def apply_name_map(df, col="participant"):
    df_copy = df.copy()  # create a copy to avoid modifying original
    df_copy[col] = df_copy[col].map(lambda x: name_map.get(x, x))
    return df_copy

df_ties_2 = apply_name_map(df_ties, "participant")
df_experience_2 = apply_name_map(df_experience, "participant")

In [10]:
df_ties_2.to_csv('../data/igem_ties_surveys/2025/survey_original_raw_data/df_ties_new.csv', index=False)
df_experience_2.to_csv('../data/igem_ties_surveys/2025/survey_original_raw_data/df_experience_new.csv', index=False)

## 2. Anonymization

In [11]:
# Dataframes: ties and experience 2025 surveys (raw data) and team rosters 2025 (scrapped data from roster API) - newest version
df_ties = pd.read_csv('../data/igem_ties_surveys/2025/survey_original_raw_data/df_ties_new.csv')
df_experience = pd.read_csv('../data/igem_ties_surveys/2025/survey_original_raw_data/df_experience_new.csv')
team_rosters_2025 = pd.read_csv('../data/igem_scrapping_2025/team_roster_2025_after_jamboree.tsv', sep='\t')

In [12]:
roster_names = team_rosters_2025["FullName"].to_list()
ties_participant_names = list(df_ties["participant"].unique())
experience_participant_names = list(df_experience["participant"].unique())

In [13]:
# Function to anonymize FullName/participant column with SHA256 hashes 

def anonymize_fullname_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    df_copy = df.copy()
    
    def hash_participant(participant: str) -> str:
        if pd.isna(participant):
            return pd.NA
        hash_object = hashlib.sha256(participant.encode('utf-8'))
        return hash_object.hexdigest()
    
    df_copy[column] = df_copy[column].apply(hash_participant)
    return df_copy

In [ ]:
# Anonymize FullName in roster data
team_rosters_2025["FullNameAnon"] = team_rosters_2025["FullName"]
team_rosters_2025 = anonymize_fullname_column(team_rosters_2025, "FullNameAnon")
team_rosters_2025 = team_rosters_2025[['FullName','FullNameAnon', 'RosterUUID', 'TeamID', 'Team', 'Year']] 
team_rosters_2025.to_csv('../data/igem_ties_surveys/2025/members_matching_table.csv') 

In [15]:
df_ties = anonymize_fullname_column(df_ties, "participant")
df_experience = anonymize_fullname_column(df_experience, "participant")

In [16]:
df_ties.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296 entries, 0 to 1295
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   survey       1296 non-null   object
 1   question     1296 non-null   object
 2   category     1296 non-null   object
 3   team         1296 non-null   object
 4   participant  1296 non-null   object
 5   answer       1144 non-null   object
dtypes: object(6)
memory usage: 60.9+ KB


In [17]:
df_ties.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296 entries, 0 to 1295
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   survey       1296 non-null   object
 1   question     1296 non-null   object
 2   category     1296 non-null   object
 3   team         1296 non-null   object
 4   participant  1296 non-null   object
 5   answer       1144 non-null   object
dtypes: object(6)
memory usage: 60.9+ KB


In [18]:
# To check if the anon names from the surveys and rosters match correctly (they should because survey names were collected from rosters)

anon_roster_names = team_rosters_2025["FullNameAnon"].to_list()
ties_participant_names = list(df_ties["participant"].unique())
experience_participant_names = list(df_experience["participant"].unique())

for name in ties_participant_names:
    if name not in anon_roster_names:
        print(f"Ties survey participant not found in roster: {name}")

for name in experience_participant_names:
    if name not in anon_roster_names:
        print(f"Experience survey participant not found in roster: {name}")

## 3. Splitting iGEM ties dataframe

In [19]:
df_ties

,survey,question,category,team,participant,answer
0,Welcome to the iGEM Team Study 2025,What is your year of birth?,inputDate,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,2003-01-01
1,Welcome to the iGEM Team Study 2025,In what country is your place of birth?,inputField,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,Mexico
2,Welcome to the iGEM Team Study 2025,Of what country are you a citizen?,inputField,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,Mexico
3,Welcome to the iGEM Team Study 2025,What gender do you identify as?,inputRadio,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,Other
4,Welcome to the iGEM Team Study 2025,What are your academic field(s) of study?,inputCheckbox,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,"Other,Life Sciences"
...,...,...,...,...,...,...
1291,Welcome to the iGEM Team Study 2025,Connection and belonging,inputTextarea,Ionis-Paris,bf624bfba404a8b1f9cbcbd08a853c53e9b9bece272cc1...,The communication with the team members
1292,Welcome to the iGEM Team Study 2025,Reflect on how your team worked together,inputTextarea,Ionis-Paris,bf624bfba404a8b1f9cbcbd08a853c53e9b9bece272cc1...,"Team building, work sessions"
1293,Welcome to the iGEM Team Study 2025,Supporting and limiting conditions,inputTextarea,Ionis-Paris,bf624bfba404a8b1f9cbcbd08a853c53e9b9bece272cc1...,The personal time was limited
1294,Welcome to the iGEM Team Study 2025,Beyond iGEM,inputTextarea,Ionis-Paris,bf624bfba404a8b1f9cbcbd08a853c53e9b9bece272cc1...,Scientific curiosité open minded


In [20]:
df_ties["team"].unique()

array(['Sorbonne University', 'Evry-Paris-Saclay', 'MSP-Maastricht',
       'Ionis-Paris', 'Aix-Marseille', 'LYON', 'Toulouse-INSA-UT', 'UGA'],
      dtype=object)

In [21]:
df_ties['question'].unique()

array(['What is your year of birth?',
       'In what country is your place of birth?',
       'Of what country are you a citizen?',
       'What gender do you identify as?',
       'What are your academic field(s) of study?',
       'Which of your 2025 team members did you know before committing to participate in the 2025 iGEM competition as a member of your team?',
       'How many years had you known this teammate for at the start of the competition?',
       'What are your other field(s) of study?',
       'How close did you feel to this person at the start of the competition?',
       'What motivated you to participate in iGEM this year?',
       'Task certainty', 'Task experience', 'Task aspiration',
       'Connection and belonging',
       'Reflect on how your team worked together',
       'Supporting and limiting conditions', 'Beyond iGEM',
       "If you'd like, share one word or phrase that captures what iGEM meant to you:"],
      dtype=object)

In [22]:
# Function to clean newlines and replace "nan" strings with real nulls in answers
def clean_text_columns(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:

    null_like = ["nan", "NaN", "NAN", "none", "None", "NULL", "null", "", " "]
    
    for c in cols:
        if c in df.columns:
            df[c] = (
                df[c]
                .astype("string") 
                .str.replace(r"\s*\n\s*", " ", regex=True)  
                .str.strip()
                # .str.replace(r'^[\'"]+|[\'"]+$', '', regex=True) # remove surrounding quotes 
                .replace(null_like, pd.NA)
            )
    return df

In [23]:
df_ties = clean_text_columns(df_ties, ['answer'])

Each of the questions from the previous output belongs to one of the 3 parts of the *iGEM ties* survey or to the *iGEM experience* survey. We will split df_ties into 4 different dataframes and continue cleaning them separately. 

In [24]:
# Drop unnecessary columns
df_clean = df_ties.drop(columns=["survey", "category"], errors="ignore")

# Questions for each df
igem_q = {
    "Connection and belonging",
    "Reflect on how your team worked together",
    "Beyond iGEM",
    "Supporting and limiting conditions",
    "If you'd like, share one word or phrase that captures what iGEM meant to you:",
}

demo_q = {
    "What is your year of birth?",
    "In what country is your place of birth?",
    "Of what country are you a citizen?",
    "What gender do you identify as?",
    "What are your academic field(s) of study?",
    "What are your other field(s) of study?",
    "What motivated you to participate in iGEM this year?",
}

relations_q = {
    "Which of your 2025 team members did you know before committing to participate in the 2025 iGEM competition as a member of your team?",
    "How many years had you known this teammate for at the start of the competition?",
    "How close did you feel to this person at the start of the competition?",
}

tasks_q = {
    "Task certainty",
    "Task experience",
    "Task aspiration",
}

# Split dataframes
igem_experience_survey = df_clean[df_clean["question"].isin(igem_q)].copy()
survey_demographics   = df_clean[df_clean["question"].isin(demo_q)].copy()
survey_relations      = df_clean[df_clean["question"].isin(relations_q)].copy()
survey_tasks          = df_clean[df_clean["question"].isin(tasks_q)].copy()


In [25]:
igem_experience_survey.shape[0] + survey_demographics.shape[0] + survey_relations.shape[0] + survey_tasks.shape[0]

1296

## 4. iGEM experience survey

In [26]:
df_experience

,question,team,participant,answer
0,Connection and belonging,DTU-Denmark,c6ac90a92af76ad0df628f1e34393cdbc073c14a6c897a...,When we did public outreach together
1,Reflect on how your team worked together,DTU-Denmark,c6ac90a92af76ad0df628f1e34393cdbc073c14a6c897a...,"We were 24, so it wad somtimes hard to collabo..."
2,Beyond iGEM,DTU-Denmark,c6ac90a92af76ad0df628f1e34393cdbc073c14a6c897a...,Better at making presentations and public spea...
3,Supporting and limiting conditions,DTU-Denmark,c6ac90a92af76ad0df628f1e34393cdbc073c14a6c897a...,NaN
4,"If you'd like, share one word or phrase that c...",DTU-Denmark,c6ac90a92af76ad0df628f1e34393cdbc073c14a6c897a...,Community
...,...,...,...,...
1417,Supporting and limiting conditions,SJTU-BioX-Shanghai,152eb32598d835441fc6dd74e538ec241b1e102cbfb4ae...,NaN
1418,"If you'd like, share one word or phrase that c...",SJTU-BioX-Shanghai,152eb32598d835441fc6dd74e538ec241b1e102cbfb4ae...,Perfect
1419,Beyond iGEM,TJUSLS-China,7a50454828260559f861a70cd32083ef56a7186e43f29d...,It made me realize that even when the surround...
1420,Supporting and limiting conditions,TJUSLS-China,7a50454828260559f861a70cd32083ef56a7186e43f29d...,When people around me misunderstand the work o...


In [27]:
# Clean newlines and replace "nan" strings with real nulls in answers
df_experience = clean_text_columns(df_experience, ['answer'])

In [28]:
df_experience.shape

(1422, 4)

In [29]:
# Merge iGEM experience survey with French teams iGEM experience data
igem_experience_survey = pd.merge(
    igem_experience_survey,
    df_experience,
    how="outer"
)

In [30]:
igem_experience_survey.shape

(1781, 4)

Now we want each question to be a column with answers as its values in the new dataframe and each person to be a row.

In [31]:
igem_experience_survey.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1781 entries, 0 to 1780
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   question     1781 non-null   object
 1   team         1781 non-null   object
 2   participant  1781 non-null   object
 3   answer       1538 non-null   string
dtypes: object(3), string(1)
memory usage: 55.8+ KB


In [32]:
# Pivot: each person is a row, each question is a column
igem_experience_survey_cleaned = (
    igem_experience_survey
    .pivot_table(
        index=["team", "participant"],
        columns="question",
        values="answer",
        aggfunc=lambda x: " | ".join(pd.Series(x).dropna().astype(str).unique())
    )
    .reset_index()
)

igem_experience_survey_cleaned.columns.name = None #remove "question" from columns index

# Rename columns
igem_experience_survey_cleaned = igem_experience_survey_cleaned.rename(columns={
    "team": "Team",
    "participant": "Participant",
    "If you'd like, share one word or phrase that captures what iGEM meant to you:":
        "One word or phrase that captures what iGEM meant to you"
})

desired_order = [
    "Team",
    "Participant",
    "Connection and belonging",
    "Reflect on how your team worked together",
    "Beyond iGEM",
    "Supporting and limiting conditions",
    "One word or phrase that captures what iGEM meant to you",
]


igem_experience_survey_cleaned = igem_experience_survey_cleaned[desired_order]

# Replace various nan strings with real nulls (pd.NA) - already done for df_ties and df_experience before but they get stringified again during pivoting
text_cols = [c for c in igem_experience_survey_cleaned.columns if c not in ("Team", "Participant")]
igem_experience_survey_cleaned = clean_text_columns(igem_experience_survey_cleaned, text_cols)

In [33]:
igem_experience_survey_cleaned

,Team,Participant,Connection and belonging,Reflect on how your team worked together,Beyond iGEM,Supporting and limiting conditions,One word or phrase that captures what iGEM meant to you
0,ABOA,368283f8a06737f0745ff5fe5654ffe90b1533ff4d3532...,Late nights in the lab especially. Venting abo...,Everyone was very supportive and great all the...,I came to love research,"Our budget limited us, also other school work",Cool beans
1,ABOA,694080d93d82f4718c505d393509e5f2a67ff5fe6c10bf...,When our stakeholders begged us to continue ou...,"""Failure is also a result"" Communication is al...",I hope in some way because I really give this ...,Money limited my ability,<NA>
2,ABOA,833fe12f88daa8a590c48a173453219155ec78b3abcb63...,"The jamboree was a great chance to connect, wh...",Collaboration was a bit difficult as I'm from ...,I think I'll be more confident in pursuing dif...,Money,"Growth of character, broader horizons"
3,AEI Prep-Taiwan,05727cd562b6be7f4f5513086751350ccb0f80cb380d21...,<NA>,<NA>,<NA>,<NA>,<NA>
4,AFCM-Egypt,2dd01eee360380d1e42bcde4d9ef95217c8c5fe21aade2...,Yes. That makes our days,Yes all the time that happened,I will meat people that appreciate my work ver...,No,It means beautiful things
...,...,...,...,...,...,...,...
347,Westlake,1f6c80f3e9b21ffc2651b98861e60b359c840f0442b379...,very good,have talk and make plan,practice experiments skills,the time limit,opportunity
348,Westlake,f782cd47d228941bc3c2a94cfd4ae83499c2b6fed1e08f...,"way heading for Jamboree from China, during wh...",During the Synthetic of lnp the leader provide...,<NA>,Division of cooperation，the part i work on is ...,new friends
349,Yale,4432071452e3932a815d36923c85b57962cd240d37fe6d...,<NA>,<NA>,<NA>,<NA>,<NA>
350,Yale,74e0bfb170fcb4882812280b6e6e9bba1dbf8bfd42ef91...,<NA>,<NA>,<NA>,<NA>,<NA>


In [34]:
igem_experience_survey_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 352 entries, 0 to 351
Data columns (total 7 columns):
 #   Column                                                   Non-Null Count  Dtype 
---  ------                                                   --------------  ----- 
 0   Team                                                     352 non-null    object
 1   Participant                                              352 non-null    object
 2   Connection and belonging                                 322 non-null    string
 3   Reflect on how your team worked together                 308 non-null    string
 4   Beyond iGEM                                              300 non-null    string
 5   Supporting and limiting conditions                       304 non-null    string
 6   One word or phrase that captures what iGEM meant to you  285 non-null    string
dtypes: object(2), string(5)
memory usage: 19.4+ KB


In [35]:
# drop rows where all columns are null except Team and Participant
igem_experience_survey_cleaned = igem_experience_survey_cleaned.dropna(how='all', subset=[c for c in igem_experience_survey_cleaned.columns if c not in ("Team", "Participant")])

In [36]:
igem_experience_survey_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 333 entries, 0 to 348
Data columns (total 7 columns):
 #   Column                                                   Non-Null Count  Dtype 
---  ------                                                   --------------  ----- 
 0   Team                                                     333 non-null    object
 1   Participant                                              333 non-null    object
 2   Connection and belonging                                 322 non-null    string
 3   Reflect on how your team worked together                 308 non-null    string
 4   Beyond iGEM                                              300 non-null    string
 5   Supporting and limiting conditions                       304 non-null    string
 6   One word or phrase that captures what iGEM meant to you  285 non-null    string
dtypes: object(2), string(5)
memory usage: 20.8+ KB


In [37]:
igem_experience_survey_cleaned.to_csv('../data/igem_ties_surveys/2025/igem_experience_survey.csv', index=False)

## 5. Demographics

In [38]:
survey_demographics

,question,team,participant,answer
0,What is your year of birth?,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,2003-01-01
1,In what country is your place of birth?,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,Mexico
2,Of what country are you a citizen?,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,Mexico
3,What gender do you identify as?,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,Other
4,What are your academic field(s) of study?,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,"Other,Life Sciences"
...,...,...,...,...
1273,Of what country are you a citizen?,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,France
1274,What gender do you identify as?,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,Female
1275,What are your academic field(s) of study?,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,Life Sciences
1278,What are your other field(s) of study?,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,<NA>


In [39]:
# Pivot: each person is a row, each question is a column
survey_demographics_cleaned = (
    survey_demographics
    .pivot_table(
        index=["team", "participant"],
        columns="question",
        values="answer",
        aggfunc=lambda x: " | ".join(pd.Series(x).dropna().astype(str).unique())
    )
    .reset_index()
)

survey_demographics_cleaned.columns.name = None #remove "question" from columns index

# Rename columns
survey_demographics_cleaned = survey_demographics_cleaned.rename(columns={
    "team": "Team",
    "participant": "Participant",
})

desired_order_demographics = [
    "Team",
    "Participant",
    "What is your year of birth?",
    "In what country is your place of birth?",
    "Of what country are you a citizen?",
    "What gender do you identify as?",
    "What are your academic field(s) of study?",
    "What are your other field(s) of study?",
    "What motivated you to participate in iGEM this year?"
]

survey_demographics_cleaned = survey_demographics_cleaned[desired_order_demographics]

# Replace various nan strings with real nulls (pd.NA) - already done for df_ties and df_experience before but they get stringified again during pivoting
text_cols = [c for c in survey_demographics_cleaned.columns if c not in ("Team", "Participant")]
survey_demographics_cleaned = clean_text_columns(survey_demographics_cleaned, text_cols)

In [40]:
country_col = "In what country is your place of birth?"
survey_demographics_cleaned[country_col].unique()

<StringArray>
[     'France',   'Argentina',     'Comoros',     'Lebanon',      'france',
     'Georgia',      'Russia',       'Egypt',      'Turkey', 'Switzerland',
      'Taiwan',     'Germany',     'Croatia',     'Belgium',       'Spain',
     'Romania',      'Greece',     'Austria',      'Mexico',      'FRANCE',
    'Portugal',    'Belgique']
Length: 22, dtype: string

In [41]:
survey_demographics_cleaned[country_col] = (
    survey_demographics_cleaned[country_col]
    .astype("string")
    .str.strip()                         
    .str.title()                         
    .replace("Belgique", "Belgium")      
    .replace(["", " ", "nan", "NaN", "None", "NULL", "null"], pd.NA) 
)

In [42]:
other_fields_col = 'What are your other field(s) of study?'

survey_demographics_cleaned[other_fields_col].unique()

<StringArray>
[                            <NA>,         '{"Other"=>"chemistry"}',
         '{"Other"=>"Chemestry"}',         '{"Other"=>"Chemistry"}',
 '{"Other"=>"Design graphisme "}']
Length: 5, dtype: string

We want to clean answers for the "What are your other field(s) of study?" questions, so they are not dictionaries, and only include what the other field of study actually is. Afterwards, we want to replace “Other” answers within the "What are your academic field(s) of study?" column by whatever the participant responded in the "What are your other field(s) of study?" column, and drop that other column.

In [43]:
# For other fields of study: Extract only the text after "Other"=> and remove the trailing "}"

s = survey_demographics_cleaned[other_fields_col].astype("string")

# Extract on rows that match {"Other"=>"..."}
pattern = r'^\s*\{\s*"Other"\s*=>\s*"([^"]+)"\s*\}\s*$'
extracted = s.str.extract(pattern, expand=False) 
s = s.where(~s.str.match(pattern, na=False), extracted)

s = s.str.strip()

# Normalize Other variants
lower = s.str.casefold()

s = s.mask(lower.isin({"chemistry", "chemestry"}), "Chemistry")

s = s.mask(lower.isin({"design graphisme"}), "Graphic Design")

s = s.replace(["", " "], pd.NA)

survey_demographics_cleaned[other_fields_col] = s


In [44]:
other_fields_col = 'What are your other field(s) of study?'
survey_demographics_cleaned[other_fields_col].unique()

<StringArray>
[<NA>, 'Chemistry', 'Graphic Design']
Length: 3, dtype: string

In [45]:
academic_fields_col = "What are your academic field(s) of study?"
survey_demographics_cleaned[academic_fields_col].unique()

<StringArray>
[                                     'Life Sciences',
     'Engineering,Life Sciences,Information Sciences',
                               'Information Sciences',
                          'Engineering,Life Sciences',
                                        'Engineering',
 'Engineering,Life Sciences,Mathematics & Statistics',
                                         'Humanities',
               'Engineering,Mathematics & Statistics',
                    'Life Sciences,Physical Sciences',
      'Information Sciences,Mathematics & Statistics',
                                'Other,Life Sciences',
                 'Life Sciences,Information Sciences',
                            'Education,Life Sciences',
                                              'Other']
Length: 14, dtype: string

In [46]:
# Replace "Other" in academic fields with the corresponding value from other fields

def replace_other_in_fields(fields, other):
    if pd.isna(fields):
        return pd.NA

    # Split academic fields on commas, trim whitespace
    parts = [p.strip() for p in str(fields).split(",")]

    # Replace any word equal to "Other" (case-insensitive) with the other value if present
    out = []
    for p in parts:
        if p.casefold() == "other" and pd.notna(other) and str(other).strip() != "":
            out.append(str(other).strip())
        else:
            out.append(p)

    # Remove duplicates while preserving order
    seen = set()
    cleaned = []
    for p in out:
        if p and p not in seen:
            seen.add(p)
            cleaned.append(p)

    return ", ".join(cleaned) if cleaned else pd.NA

# Usage
survey_demographics_cleaned[academic_fields_col] = [
    replace_other_in_fields(a, o)
    for a, o in zip(survey_demographics_cleaned[academic_fields_col], survey_demographics_cleaned[other_fields_col])
]

# Drop the "other fields" column once merged
survey_demographics_cleaned = survey_demographics_cleaned.drop(columns=[other_fields_col])

In [47]:
survey_demographics_cleaned[academic_fields_col].unique()

array(['Life Sciences',
       'Engineering, Life Sciences, Information Sciences',
       'Information Sciences', 'Engineering, Life Sciences',
       'Engineering',
       'Engineering, Life Sciences, Mathematics & Statistics',
       'Humanities', 'Engineering, Mathematics & Statistics',
       'Life Sciences, Physical Sciences',
       'Information Sciences, Mathematics & Statistics',
       'Chemistry, Life Sciences', 'Life Sciences, Information Sciences',
       'Education, Life Sciences', 'Graphic Design'], dtype=object)

In [48]:
survey_demographics_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 8 columns):
 #   Column                                                Non-Null Count  Dtype 
---  ------                                                --------------  ----- 
 0   Team                                                  72 non-null     object
 1   Participant                                           72 non-null     object
 2   What is your year of birth?                           72 non-null     string
 3   In what country is your place of birth?               72 non-null     string
 4   Of what country are you a citizen?                    72 non-null     string
 5   What gender do you identify as?                       72 non-null     string
 6   What are your academic field(s) of study?             72 non-null     object
 7   What motivated you to participate in iGEM this year?  72 non-null     string
dtypes: object(3), string(5)
memory usage: 4.6+ KB


In [49]:
survey_demographics_cleaned.to_csv('../data/igem_ties_surveys/2025/survey_demographics.csv', index=False)

## 6. Relations

In [50]:
survey_relations

,question,team,participant,answer
5,Which of your 2025 team members did you know b...,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,Marco DA COSTA
6,How many years had you known this teammate for...,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,"{""Marco DA COSTA""=>{""""=>""Between 1 and 2 years""}}"
8,How close did you feel to this person at the s...,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,"{""Marco DA COSTA""=>{""""=>""Somewhat close""}}"
23,Which of your 2025 team members did you know b...,Sorbonne University,1d3de9a1f9a7396b9685a91b3a24bd568c087b64d89d84...,<NA>
24,How many years had you known this teammate for...,Sorbonne University,1d3de9a1f9a7396b9685a91b3a24bd568c087b64d89d84...,<NA>
...,...,...,...,...
1259,How many years had you known this teammate for...,UGA,15c50bf7d46332aff78a635fafaa458c66756f7eb11f2d...,"{""Lina ""=>{""""=>""Less than 1 year""}, ""Selma Koc..."
1261,How close did you feel to this person at the s...,UGA,15c50bf7d46332aff78a635fafaa458c66756f7eb11f2d...,"{""Lina ""=>{""""=>""Somewhat distant""}, ""Selma Koc..."
1276,Which of your 2025 team members did you know b...,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,"pierre cavailles,Sébastien,Yaceton,Lina ,Selma..."
1277,How many years had you known this teammate for...,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,"{""Lina ""=>{""""=>""More than 4 years""}, ""Yaceton""..."


In [51]:
survey_relations.info()

<class 'pandas.core.frame.DataFrame'>
Index: 216 entries, 5 to 1279
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   question     216 non-null    object
 1   team         216 non-null    object
 2   participant  216 non-null    object
 3   answer       195 non-null    string
dtypes: object(3), string(1)
memory usage: 8.4+ KB


In [52]:
# Pivot: each person is a row, each question is a column for now

survey_relations_cleaned = (
    survey_relations
    .pivot_table(
        index=["team", "participant"],
        columns="question",
        values="answer",
        aggfunc=lambda x: " | ".join(pd.Series(x).dropna().astype(str).unique())
    )
    .reset_index()
)

survey_relations_cleaned.columns.name = None

survey_relations_cleaned = survey_relations_cleaned.rename(columns={
    "team": "Team",
    "participant": "SourceParticipant",
    "How many years had you known this teammate for at the start of the competition?": "YearsKnown",
    "How close did you feel to this person at the start of the competition?": "Closeness"
})

desired_order_relations = [
    "Team",
    "SourceParticipant",
    "Which of your 2025 team members did you know before committing to participate in the 2025 iGEM competition as a member of your team?",
    "YearsKnown",
    "Closeness"
]

survey_relations_cleaned = survey_relations_cleaned[desired_order_relations]

text_cols = [c for c in survey_relations_cleaned.columns if c not in ("Team", "SourceParticipant")]
survey_relations_cleaned = clean_text_columns(survey_relations_cleaned, text_cols)

In [53]:
survey_relations_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 5 columns):
 #   Column                                                                                                                                Non-Null Count  Dtype 
---  ------                                                                                                                                --------------  ----- 
 0   Team                                                                                                                                  72 non-null     object
 1   SourceParticipant                                                                                                                     72 non-null     object
 2   Which of your 2025 team members did you know before committing to participate in the 2025 iGEM competition as a member of your team?  65 non-null     string
 3   YearsKnown                                                                         

In [54]:
survey_relations_cleaned

,Team,SourceParticipant,Which of your 2025 team members did you know before committing to participate in the 2025 iGEM competition as a member of your team?,YearsKnown,Closeness
0,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,"Luna,Ruzanna,James Sturgis,Laetitia HOUOT","{""Luna""=>{""""=>""Between 2 and 3 years""}, ""Ruzan...","{""Luna""=>{""""=>""Somewhat close""}, ""Ruzanna""=>{""..."
1,Aix-Marseille,1f02e917fd21530620990a81a8ddce4a8cb1f0e57770e3...,"Laetitia HOUOT,Miriem","{""Miriem""=>{""""=>""Between 2 and 3 years""}, ""Lae...","{""Miriem""=>{""""=>""Very close""}, ""Laetitia HOUOT..."
2,Aix-Marseille,44e0b4257b5d6473970e6332b76b5c0005e21b5d22a749...,"Hadjara ,Sandra","{""Sandra""=>{""""=>""Between 2 and 3 years""}, ""Had...","{""Sandra""=>{""""=>""Very close""}, ""Hadjara ""=>{""""..."
3,Aix-Marseille,5520135e1e537bc0bdedfd6fcadaae62196285e94cd451...,"James Sturgis,Laetitia HOUOT","{""James Sturgis""=>{""""=>""Less than 1 year""}, ""L...","{""James Sturgis""=>{""""=>""Neither close nor dist..."
4,Aix-Marseille,59c407aaa03d80adaccd1bdfb7b3fe45d8e2e2ebb24aa2...,"Luna,Manon,Miriem","{""Luna""=>{""""=>""Between 1 and 2 years""}, ""Manon...","{""Luna""=>{""""=>""Somewhat close""}, ""Manon""=>{""""=..."
...,...,...,...,...,...
67,UGA,89725d1e4dfcec8cbbd2720c0c7c5c4d56130bf75e2947...,pierre cavailles,"{""pierre cavailles""=>{""""=>""More than 4 years""}}","{""pierre cavailles""=>{""""=>""Somewhat close""}}"
68,UGA,9d34c51716b00699fd9fc45b858c5a5e7aeb8e65bcd089...,"Lina ,Chloé Chambard,pierre cavailles,Sébastien","{""Lina ""=>{""""=>""More than 4 years""}, ""Sébastie...","{""Lina ""=>{""""=>""Very close""}, ""Sébastien""=>{""""..."
69,UGA,a2b48b51d07f52ee9d7d8f2a74a6c61895b17870a1badf...,"Guillaume Banon,Heloise,Thibaut","{""Heloise""=>{""""=>""Between 1 and 2 years""}, ""Th...","{""Heloise""=>{""""=>""Somewhat close""}, ""Thibaut""=..."
70,UGA,abcdf154b6aaeabd3361ad73c5a1d6dcd1e549f476520e...,"Heloise,Thibaut,Matthieu","{""Heloise""=>{""""=>""Between 1 and 2 years""}, ""Th...","{""Heloise""=>{""""=>""Very close""}, ""Thibaut""=>{""""..."


In [55]:
# survey_relations_cleaned["YearsKnown"].unique()

In [56]:
target_members_col = "Which of your 2025 team members did you know before committing to participate in the 2025 iGEM competition as a member of your team?"
# survey_relations_cleaned[target_members_col].unique()

In [57]:
# survey_relations_cleaned["Closeness"].unique()

Team member names should be extracted from string array values in the "Which of your 2025 team members did you know before committing to participate in the 2025 iGEM competition as a member of your team?" column. Each name is separated by a comma there. This column should be deleted, and a new column called "TargetParticipant" should be added. In this dataframe, each row represents a (SourceParticipant, TargetParticipant) pair, while source participants have as many rows as they have connections. For each of the target participants, closeness and years known values should be extracted from dictionaries in those columns.

In [58]:
# Helper functions

# Function to parse strings like {"Luna"=>{""=>"Between 1 and 2 years"}} into Python dicts like {'Luna': 'Between 1 and 2 years'}
def parse_string_like_dict(s: str) -> dict:
    import json, re
    if not isinstance(s, str) or not s.strip():
        return {}
    txt = s.replace("=>", ":")
    try:
        d = json.loads(txt)
    except Exception:
        # fallback cleanup for weird spaces
        txt = re.sub(r"\s*:\s*", ":", txt)
        txt = re.sub(r"\s*,\s*", ",", txt)
        try:
            d = json.loads(txt)
        except Exception:
            return {}
    # Flatten if inner dicts are like {"": "value"}
    flat = {}
    for k, v in d.items():
        if isinstance(v, dict) and "" in v:
            flat[k] = v[""]
        else:
            flat[k] = v
    return flat


# Function to extract mapped value for a target from a mapping string
def extract_mapped_value(mapping_str: object, target: str):
    d = parse_string_like_dict(mapping_str) if isinstance(mapping_str, str) else {}
    if not d or target not in d:
        return pd.NA
    val = d[target]
    if isinstance(val, dict):
        # take the first non-null value
        for v in val.values():
            if v not in (None, "", " ", "nan", "NaN"):
                return v
        return pd.NA
    return val if val not in ("", " ", None) else pd.NA

# Function to split the team members string array into a list of names
def split_targets(s: object):
    if s is pd.NA or s is None:
        return []
    if not isinstance(s, str):
        return []
    parts = [p.strip() for p in s.split(',')]
    parts = [re.sub(r'\s+', ' ', p).strip().strip(',') for p in parts]
    return [p for p in parts if p]  


In [59]:
# Example usages 

print(parse_string_like_dict('{"Luna"=>{""=>"Between 1 and 2 years"}}'))
print(extract_mapped_value('{"Luna"=>{""=>"Between 1 and 2 years"}, "Manon"=>{""=>"Between 2 and 3 years"}}', "Luna"))
print(split_targets("Luna, Manon , Miriem"))

{'Luna': 'Between 1 and 2 years'}
Between 1 and 2 years
['Luna', 'Manon', 'Miriem']


In [60]:
def make_survey_relations_final(survey_relations_cleaned: pd.DataFrame) -> pd.DataFrame:
    df = survey_relations_cleaned.copy()

    # Normalize key columns to string type 
    for c in ["Team", "SourceParticipant", target_members_col, "YearsKnown", "Closeness"]:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # Create TargetParticipant
    if "TargetParticipant" not in df.columns:
        df["TargetParticipant"] = df[target_members_col].apply(split_targets)
        df = df.explode("TargetParticipant", ignore_index=True)

    # Clean TargetParticipant 
    df["TargetParticipant"] = (
        df["TargetParticipant"]
        .astype("string")
        .str.strip()
        .replace(["", " "], pd.NA)
    )

    # Extract mapped values for YearsKnown and Closeness using the current target
    df["YearsKnown"] = df.apply(
        lambda r: extract_mapped_value(r.get("YearsKnown", pd.NA), r["TargetParticipant"])
                  if pd.notna(r["TargetParticipant"]) else pd.NA,
        axis=1
    ).astype("string")

    df["Closeness"] = df.apply(
        lambda r: extract_mapped_value(r.get("Closeness", pd.NA), r["TargetParticipant"])
                  if pd.notna(r["TargetParticipant"]) else pd.NA,
        axis=1
    ).astype("string")

    # Drop the original long question column if present
    if target_members_col in df.columns:
        df = df.drop(columns=[target_members_col])

    # Reorder columns 
    desired = ["Team", "SourceParticipant", "TargetParticipant", "YearsKnown", "Closeness"]
    existing = [c for c in desired if c in df.columns]
    others = [c for c in df.columns if c not in existing]
    df = df[existing + others]

    return df

survey_relations_final = make_survey_relations_final(survey_relations_cleaned)


In [61]:
survey_relations_final

,Team,SourceParticipant,TargetParticipant,YearsKnown,Closeness
0,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,Luna,Between 2 and 3 years,Somewhat close
1,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,Ruzanna,Between 1 and 2 years,Neither close nor distant
2,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,James Sturgis,Between 2 and 3 years,Very distant
3,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,Laetitia HOUOT,More than 4 years,Very distant
4,Aix-Marseille,1f02e917fd21530620990a81a8ddce4a8cb1f0e57770e3...,Laetitia HOUOT,Less than 1 year,Very distant
...,...,...,...,...,...
291,UGA,abcdf154b6aaeabd3361ad73c5a1d6dcd1e549f476520e...,Thibaut,Between 1 and 2 years,Somewhat close
292,UGA,abcdf154b6aaeabd3361ad73c5a1d6dcd1e549f476520e...,Matthieu,Between 1 and 2 years,Somewhat close
293,UGA,e241e0788f28689c854db0f5521233b34408e71644dcf5...,Guillaume Banon,Between 1 and 2 years,Somewhat close
294,UGA,e241e0788f28689c854db0f5521233b34408e71644dcf5...,Heloise,Between 1 and 2 years,Somewhat close


In [62]:
survey_relations_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296 entries, 0 to 295
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Team               296 non-null    string
 1   SourceParticipant  296 non-null    string
 2   TargetParticipant  289 non-null    string
 3   YearsKnown         272 non-null    string
 4   Closeness          272 non-null    string
dtypes: string(5)
memory usage: 11.7 KB


In [63]:
survey_relations_final["YearsKnown"].unique()

<StringArray>
['Between 2 and 3 years', 'Between 1 and 2 years',     'More than 4 years',
      'Less than 1 year',                    <NA>]
Length: 5, dtype: string

In [64]:
survey_relations_final["Closeness"].unique()

<StringArray>
[           'Somewhat close', 'Neither close nor distant',
              'Very distant',                'Very close',
                        <NA>,          'Somewhat distant']
Length: 6, dtype: string

In [65]:
# Print rows where Target is NA
survey_relations_final[survey_relations_final["TargetParticipant"].isna()]

,Team,SourceParticipant,TargetParticipant,YearsKnown,Closeness
44,Evry-Paris-Saclay,567cac7911ebe43f4c2a3e8ccf480acfede6dd1c1757c3...,<NA>,<NA>,<NA>
56,Evry-Paris-Saclay,84844ac16946ec4ab84a3c4a6d6a345b58faf0252f29ce...,<NA>,<NA>,<NA>
78,Ionis-Paris,bf624bfba404a8b1f9cbcbd08a853c53e9b9bece272cc1...,<NA>,<NA>,<NA>
192,MSP-Maastricht,ccde50f884a353ee364dd1e8fa6fb7cad7ed9ec26866cc...,<NA>,<NA>,<NA>
208,Sorbonne University,1d3de9a1f9a7396b9685a91b3a24bd568c087b64d89d84...,<NA>,<NA>,<NA>
252,Toulouse-INSA-UT,35646ad04e82b53709ff3defa061f8cf87e7b6e93411c1...,<NA>,<NA>,<NA>
272,UGA,174326d01434105a00dce65350b6cc9ece6f20fc6dc1e5...,<NA>,<NA>,<NA>


In [66]:
survey_relations_final = survey_relations_final.dropna(subset=["TargetParticipant"])

In [67]:
survey_relations_final.to_csv('../data/igem_ties_surveys/2025/survey_relations.csv', index=False)

## 7. Tasks

In [68]:
survey_tasks

,question,team,participant,answer
10,Task certainty,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,"{""Conceptualization: Brainstorming and develop..."
11,Task experience,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,"{""Conceptualization: Brainstorming and develop..."
12,Task aspiration,Sorbonne University,b53fc9ab37dc72c5fba725ed893d52582471aa087385b6...,"{""Conceptualization: Brainstorming and develop..."
28,Task certainty,Sorbonne University,1d3de9a1f9a7396b9685a91b3a24bd568c087b64d89d84...,"{""Conceptualization: Brainstorming and develop..."
31,Task experience,Sorbonne University,1d3de9a1f9a7396b9685a91b3a24bd568c087b64d89d84...,"{""Conceptualization: Brainstorming and develop..."
...,...,...,...,...
1281,Task certainty,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,"{""Conceptualization: Brainstorming and develop..."
1282,Task experience,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,"{""Conceptualization: Brainstorming and develop..."
1283,Task aspiration,UGA,4b295471904fa0455e7228267fb5575ab5a6152b87fc26...,"{""Conceptualization: Brainstorming and develop..."
1289,Task experience,Ionis-Paris,bf624bfba404a8b1f9cbcbd08a853c53e9b9bece272cc1...,"{""Conceptualization: Brainstorming and develop..."


In [69]:
survey_tasks.info()

<class 'pandas.core.frame.DataFrame'>
Index: 216 entries, 10 to 1290
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   question     216 non-null    object
 1   team         216 non-null    object
 2   participant  216 non-null    object
 3   answer       216 non-null    string
dtypes: object(3), string(1)
memory usage: 8.4+ KB


In [70]:
# Pivot: each person is a row, each question is a column for now

survey_tasks_cleaned = (
    survey_tasks
    .pivot_table(
        index=["team", "participant"],
        columns="question",
        values="answer",
        aggfunc=lambda x: " | ".join(pd.Series(x).dropna().astype(str).unique())
    )
    .reset_index()
)

survey_tasks_cleaned.columns.name = None

survey_tasks_cleaned = survey_tasks_cleaned.rename(columns={
    "team": "Team",
    "participant": "Participant",
    "Task certainty": "Certainty",
    "Task experience": "Experience",
    "Task aspiration": "Preference"
})

desired_order_tasks = [
    "Team",
    "Participant",
    "Certainty",
    "Experience",
    "Preference"
]

survey_tasks_cleaned = survey_tasks_cleaned[desired_order_tasks]

task_cols = [c for c in survey_tasks_cleaned.columns if c not in ("Team", "Participant")]
survey_tasks_cleaned = clean_text_columns(survey_tasks_cleaned, task_cols)

In [71]:
survey_tasks_cleaned

,Team,Participant,Certainty,Experience,Preference
0,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."
1,Aix-Marseille,1f02e917fd21530620990a81a8ddce4a8cb1f0e57770e3...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."
2,Aix-Marseille,44e0b4257b5d6473970e6332b76b5c0005e21b5d22a749...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."
3,Aix-Marseille,5520135e1e537bc0bdedfd6fcadaae62196285e94cd451...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."
4,Aix-Marseille,59c407aaa03d80adaccd1bdfb7b3fe45d8e2e2ebb24aa2...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."
...,...,...,...,...,...
67,UGA,89725d1e4dfcec8cbbd2720c0c7c5c4d56130bf75e2947...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."
68,UGA,9d34c51716b00699fd9fc45b858c5a5e7aeb8e65bcd089...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."
69,UGA,a2b48b51d07f52ee9d7d8f2a74a6c61895b17870a1badf...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."
70,UGA,abcdf154b6aaeabd3361ad73c5a1d6dcd1e549f476520e...,"{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop...","{""Conceptualization: Brainstorming and develop..."


In [72]:
survey_tasks_cleaned.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Team         72 non-null     object
 1   Participant  72 non-null     object
 2   Certainty    72 non-null     string
 3   Experience   72 non-null     string
 4   Preference   72 non-null     string
dtypes: object(2), string(3)
memory usage: 2.9+ KB


In [73]:
# survey_tasks_cleaned['Certainty'].unique()

There are 15 tasks (without "Wiki Coding" and "Other" this year) for which the participants rated their certainty, experience, and preference. We want to make a new dataframe with one row per (Participant, Task) pair. Similarly as before, we will extract certainty, experience, and preverence values for each task for each person. Task names will be the values of the Task column.

In [74]:
# The 15 tasks for this year (without "Wiki Coding" and "Other"):
tasks = [
    "Conceptualization: Brainstorming and developing ideas for the project", 
    "Investigation: Performing the experiments and/or collecting data/evidence", 
    "Software: Developing, implementing, and/or testing computer programs and code", 
    "Lab Maintenance: Cleaning, organizing, and preparing the research facilities or lab", 
    "Hardware: Designing, building, and/or testing mechanical, electrical, or optical hardware systems", 
    "Project Administration: Managing and coordinating the project activities, planning, and execution", 
    "Public Engagement: Preparing and/or implementing tools/activities to engage the broader community", 
    "Entrepreneurship: Developing business models and other commercially relevant materials or activities", 
    "Fundraising: Raising money, in-kind donations of materials, and/or services for the team’s activities", 
    "Writing: Writing, reviewing or editing content for the wiki or other documents to be shared outside the team", 
    "Background Research: reading peer-reviewed scientific research articles, white papers, regulations/guidelines, and other documents", 
    "Safety: Performing activities to ensure compliance with the safety guidelines or requirements of iGEM, your institution, and/or government", 
    "Analysis: analyzing data generated by your team, another iGEM team (current or previous), and/or from literature as needed for your project", 
    "Visualization: Preparation, creation and/or presentation of the work including data visualization, user interfaces, videos and other graphics", 
    "Data Curation: Annotating (producing metadata), cleaning, and maintaining research data for your team’s project and for future use by other iGEM teams or members of the synthetic biology community"
]

# Task names mapping (without explanation that is presented in the survey)
short_map = {
   tasks[0]:  "Conceptualization",
   tasks[1]:  "Investigation",
   tasks[2]:  "Software",
   tasks[3]:  "Lab Maintenance",
   tasks[4]:  "Hardware",
   tasks[5]:  "Project Administration",
   tasks[6]:  "Public Engagement",
   tasks[7]:  "Entrepreneurship",
   tasks[8]:  "Fundraising",
   tasks[9]:  "Writing",
   tasks[10]: "Background Research",
   tasks[11]: "Safety",
   tasks[12]: "Analysis",
   tasks[13]: "Visualization",
   tasks[14]: "Data Curation",
}

In [75]:
# Example usages of previous helper functions

string = '{"Conceptualization: Brainstorming and developing ideas for the project"=>"Completely sure", "Investigation: Performing the experiments and/or collecting data/evidence"=>"Completely sure", "Software: Developing, implementing, and/or testing computer programs and code"=>"Completely sure", "Lab Maintenance: Cleaning, organizing, and preparing the research facilities or lab"=>"Completely sure", "Hardware: Designing, building, and/or testing mechanical, electrical, or optical hardware systems"=>"Completely unsure", "Project Administration: Managing and coordinating the project activities, planning, and execution"=>"Completely sure", "Public Engagement: Preparing and/or implementing tools/activities to engage the broader community"=>"Completely sure", "Entrepreneurship: Developing business models and other commercially relevant materials or activities"=>"Somewhat sure", "Fundraising: Raising money, in-kind donations of materials, and/or services for the team’s activities"=>"Completely sure", "Writing: Writing, reviewing or editing content for the wiki or other documents to be shared outside the team"=>"Completely sure", "Background Research: reading peer-reviewed scientific research articles, white papers, regulations/guidelines, and other documents"=>"Completely sure", "Safety: Performing activities to ensure compliance with the safety guidelines or requirements of iGEM, your institution, and/or government"=>"Completely sure", "Analysis: analyzing data generated by your team, another iGEM team (current or previous), and/or from literature as needed for your project"=>"Completely sure", "Visualization: Preparation, creation and/or presentation of the work including data visualization, user interfaces, videos and other graphics"=>"Completely sure", "Data Curation: Annotating (producing metadata), cleaning, and maintaining research data for your team’s project and for future use by other iGEM teams or members of the synthetic biology community"=>"Completely sure"}'

print(parse_string_like_dict(string))

print(extract_mapped_value(string, "Conceptualization: Brainstorming and developing ideas for the project"))

{'Conceptualization: Brainstorming and developing ideas for the project': 'Completely sure', 'Investigation: Performing the experiments and/or collecting data/evidence': 'Completely sure', 'Software: Developing, implementing, and/or testing computer programs and code': 'Completely sure', 'Lab Maintenance: Cleaning, organizing, and preparing the research facilities or lab': 'Completely sure', 'Hardware: Designing, building, and/or testing mechanical, electrical, or optical hardware systems': 'Completely unsure', 'Project Administration: Managing and coordinating the project activities, planning, and execution': 'Completely sure', 'Public Engagement: Preparing and/or implementing tools/activities to engage the broader community': 'Completely sure', 'Entrepreneurship: Developing business models and other commercially relevant materials or activities': 'Somewhat sure', 'Fundraising: Raising money, in-kind donations of materials, and/or services for the team’s activities': 'Completely sure'

In [76]:
# Final function to make each task a separate row per (Participant, Task) - one participant belongs to just one team anyways

def make_survey_tasks_final(survey_tasks_cleaned: pd.DataFrame) -> pd.DataFrame:
    df = survey_tasks_cleaned.copy()

    # Normalize key columns
    for c in ["Team", "Participant", "Certainty", "Experience", "Preference"]:
        if c in df.columns:
            df[c] = df[c].astype("string")

    if "Task" not in df.columns:
        # attach full task list to each row, then explode
        df["Task"] = [tasks] * len(df)
        df = df.explode("Task", ignore_index=True)
    else:
        # if Task exists but contains lists, explode; if already exploded (strings), leave as-is
        if df["Task"].apply(lambda x: isinstance(x, (list, tuple))).any():
            df = df.explode("Task", ignore_index=True)

    # Fill Certainty, Experience, and Preference per task (null if the task key is missing)
    for col in ["Certainty", "Experience", "Preference"]:
        if col in df.columns:
            df[col] = df.apply(
                lambda r: extract_mapped_value(r.get(col, pd.NA), r["Task"])
                          if pd.notna(r["Task"]) else pd.NA,
                axis=1
            ).astype("string")

    # Rename tasks to short labels
    df["Task"] = df["Task"].map(short_map).astype("string")

    # Reorder columns
    desired = ["Team", "Participant", "Task", "Certainty", "Experience", "Preference"]
    existing = [c for c in desired if c in df.columns]
    df = df[existing + [c for c in df.columns if c not in existing]]

    return df

survey_tasks_final = make_survey_tasks_final(survey_tasks_cleaned)

In [77]:
survey_tasks_final

,Team,Participant,Task,Certainty,Experience,Preference
0,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,Conceptualization,Somewhat sure,Between three and four years,Somewhat agree
1,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,Investigation,Neutral,Less than one year,Somewhat agree
2,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,Software,Somewhat unsure,Less than one year,Strongly agree
3,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,Lab Maintenance,Completely sure,Less than one year,Strongly agree
4,Aix-Marseille,190987e7c02f48b91221bc4914ee26eaec486dab371296...,Hardware,Completely unsure,Less than one year,Strongly disagree
...,...,...,...,...,...,...
1075,UGA,e241e0788f28689c854db0f5521233b34408e71644dcf5...,Background Research,Neutral,Less than one year,Neutral
1076,UGA,e241e0788f28689c854db0f5521233b34408e71644dcf5...,Safety,Somewhat sure,Less than one year,Neutral
1077,UGA,e241e0788f28689c854db0f5521233b34408e71644dcf5...,Analysis,Somewhat sure,Less than one year,Neutral
1078,UGA,e241e0788f28689c854db0f5521233b34408e71644dcf5...,Visualization,Somewhat sure,Less than one year,Neutral


In [78]:
survey_tasks_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1080 entries, 0 to 1079
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Team         1080 non-null   string
 1   Participant  1080 non-null   string
 2   Task         1080 non-null   string
 3   Certainty    1080 non-null   string
 4   Experience   1080 non-null   string
 5   Preference   1080 non-null   string
dtypes: string(6)
memory usage: 50.8 KB


In [79]:
survey_tasks_final["Certainty"].unique()

<StringArray>
[    'Somewhat sure',           'Neutral',   'Somewhat unsure',
   'Completely sure', 'Completely unsure']
Length: 5, dtype: string

In [80]:
survey_tasks_final["Experience"].unique()

<StringArray>
['Between three and four years',           'Less than one year',
    'Between one and two years',  'Between two and three years',
  'Between four and five years',   'Between five and six years',
          'More than six years']
Length: 7, dtype: string

In [81]:
survey_tasks_final["Preference"].unique()

<StringArray>
[   'Somewhat agree',    'Strongly agree', 'Strongly disagree',
           'Neutral', 'Somewhat disagree']
Length: 5, dtype: string

In [82]:
survey_tasks_final.to_csv('../data/igem_ties_surveys/2025/survey_tasks.csv', index=False)

## 8. Merge tasks survey and attributions

We will reuse code from the `3_connect_survey_tasks_with_attributions.ipynb` notebook to connect task survey responses fith the attributions df, by adding a binary `TaskPerformed` column to the survey_tasks_final dataframe.

In [83]:
# Check if now anon attributions name match anon task survey names (after we already replaced old roster names with new ones)

attributions_2025 = pd.read_csv('../results/attributions/results_attributions_2025.tsv', sep='\t')
attributions_2025 = anonymize_fullname_column(attributions_2025, "FullName")

attribution_names = list(attributions_2025["FullName"].unique())
tasks_participant_names = list(survey_tasks_final["Participant"].unique()) # anon names in survey

not_found_names = []
for name in tasks_participant_names:
    if name not in attribution_names:
        not_found_names.append(name)
print(len(not_found_names), not_found_names)

0 []


In [84]:
attributions_2025.to_csv("../data/attributions/2025_attributions/attributions_anonymized_2025.tsv", sep="\t", index=False)

After old participant names are replaced with the new ones (fetched from roster API after the jamboree, subtitle 1), all task survey participant names have a match in the attributions dataframe. That means that we can now match these two dataframes by adding a `TaskPerformed` column.

In [85]:
# Clean Task column
 
'''
- Convert Task to lowercase
- Replace dashes with spaces
- Strip leading and trailing spaces
- Change some task names to fit the survey
'''

def clean_task_name(df):
    if "Task" in df.columns:
        df = df.copy()
        df["Task"] = (
            df["Task"]
            .astype(str)
            .str.lower()
            .str.replace("-", " ", regex=False)
            .str.strip()
        )

        # Specific name normalizations
        df["Task"] = (
            df["Task"]
            .replace({
                "software development": "software",
                "outreach & education": "public engagement",
                "conceptualization  other": "conceptualization",
                "conceptualisation": "conceptualization",
                "notebook keeping": "writing",
                "wiki coding": "software"
            })
        )
    return df

In [86]:
attributions_2025 = clean_task_name(attributions_2025)
# attributions_2025.head()

In [87]:
survey_tasks_final = clean_task_name(survey_tasks_final)
# survey_tasks_final.head()

In [88]:
def combine_survey_tasks_with_attributions(survey_df, attributions_df,
                                           s_task_col, a_task_col,
                                           s_participant_col, a_fullname_col):
    """
    Combine survey tasks with attributions, adding 'TaskPerformed' and 'Role' columns gathered from attributions.
    
    Parameters:
    - survey_df (DataFrame): dataframe with survey task responses
    - attributions_df (DataFrame): dataframe with team attributions
    - s_task_col (str): column name in survey_df for task names ('Task')
    - a_task_col (str): column name in attributions_df for task names ('Task')
    - s_participant_col (str): column name in survey_df for participant hash ('participant')
    - a_fullname_col (str): column name in attributions_df for participant hash ('Participant')

    Returns:
    - df: a copy of survey_df with an additional 'TaskPerformed' column:
        - True: participant-task pair exists in attributions
        - False: participant exists, but task not TaskPerformed
        - NA: participant not found in attributions
    and 'Role' column - the participant's role from attributions_df, matched by participant name.
    """
    df = survey_df.copy()
    attributions_copy = attributions_df.copy()

    # Create merge key for exact participant-task match
    df['_merge_key'] = df[s_participant_col] + '||' + df[s_task_col]
    attributions_copy['_merge_key'] = attributions_copy[a_fullname_col] + '||' + attributions_copy[a_task_col]

    # Assign TaskPerformed based on composite key
    df['TaskPerformed'] = df['_merge_key'].isin(attributions_copy['_merge_key']).astype('Int64')

    # Set TaskPerformed = pd.NA where participant is not found in attributions
    unmatched_participants = ~df[s_participant_col].isin(attributions_copy[a_fullname_col])
    df.loc[unmatched_participants, 'TaskPerformed'] = pd.NA

    # Add Role by merging on participant only (not task), avoiding column collision
    roles = attributions_copy[[a_fullname_col, 'Role']].drop_duplicates(subset=[a_fullname_col])
    roles = roles.rename(columns={'Role': '_Role'})  # temporary name to avoid conflict
    df = df.merge(roles, left_on=s_participant_col, right_on=a_fullname_col, how='left')
    df.rename(columns={'_Role': 'Role'}, inplace=True)

    # Cleanup
    df.drop(columns=['_merge_key', a_fullname_col], inplace=True)

    return df


In [89]:
combined_df = combine_survey_tasks_with_attributions(
    survey_df=survey_tasks_final,
    attributions_df=attributions_2025,
    s_task_col='Task',
    a_task_col='Task',
    s_participant_col='Participant',
    a_fullname_col='FullName'
)

In [90]:
combined_df.rename(columns={'Participant': 'TeamMember'}, inplace=True)

In [91]:
# Total number of rows
total_rows = len(combined_df)

# Number of rows where TaskPerformed == 0
task_not_performed = (combined_df['TaskPerformed'] == 0).sum()

# Number of rows where TaskPerformed == 1
task_performed = (combined_df['TaskPerformed'] == 1).sum()

# Number of rows where TaskPerformed is null 
task_performed_null = combined_df['TaskPerformed'].isna().sum()

print(f"Total rows in combined dataframe: {total_rows}")
print(f"TaskPerformed = 0: {task_not_performed}")
print(f"TaskPerformed = 1: {task_performed}")
print(f"TaskPerformed = NA: {task_performed_null}")

Total rows in combined dataframe: 1080
TaskPerformed = 0: 613
TaskPerformed = 1: 467
TaskPerformed = NA: 0


In [92]:
# Cleanup of the column names and values - Run with final results

# Capitalize the first letter of each Task entry
combined_df['Task'] = combined_df['Task'].str.title()

# Replace dashes with spaces in Role, then capitalize first letter
combined_df['Role'] = (
    combined_df['Role']
    .str.replace('-', ' ')
    .str.title()
)

combined_df['Role'] = (
    combined_df['Role']
    .replace({
        'Student': 'Student Member',
        'Primary Pi': 'Primary PI',
        'Secondary Pi': 'Secondary PI'
    })
)

# Rename the “team” column to “Team”
combined_df.rename(columns={'team': 'Team'}, inplace=True)

# Reorder columns
new_column_order = [
    "Team",
    "TeamMember",  
    "Role", 
    "Task", 
    "Certainty",
    "Experience",
    "Preference",
    "TaskPerformed"
]
combined_df = combined_df[new_column_order]

# Save as TSV
combined_df.to_csv("../results/survey_tasks_and_attributions/2025_survey_tasks_and_attributions_combined.tsv", sep="\t", index=False)

In [93]:
# Tasks that exist in attributions but not in the task survey

attributions_2025_clean = clean_task_name(attributions_2025)
combined_df_clean = clean_task_name(combined_df)

missing_tasks = (
    attributions_2025_clean.loc[
        ~attributions_2025_clean["Task"].isin(combined_df_clean["Task"])
    ]["Task"]
    .dropna()
    .unique()
)

print("Tasks in attributions but not in task survey:")
for task in missing_tasks:
    print("-", task)

Tasks in attributions but not in task survey:
- other
- validation
- supervision
- drylab modeling


In [94]:
combined_df["Task"].unique()

array(['Conceptualization', 'Investigation', 'Software',
       'Lab Maintenance', 'Hardware', 'Project Administration',
       'Public Engagement', 'Entrepreneurship', 'Fundraising', 'Writing',
       'Background Research', 'Safety', 'Analysis', 'Visualization',
       'Data Curation'], dtype=object)

In [95]:
attributions_2025["Task"].unique()

array(['conceptualization', 'other', 'safety', 'data curation',
       'analysis', 'public engagement', 'background research',
       'fundraising', 'investigation', 'writing',
       'project administration', 'visualization', 'software',
       'entrepreneurship', 'hardware', 'validation', 'supervision',
       'drylab modeling'], dtype=object)

In [96]:
attributions_2025[attributions_2025["Task"].isin(missing_tasks)]

,FullName,Username,UUID,TeamID,Team,Year,Role,Task,TaskDescription
1,61d84c4c029d57713da1f015ca5a5e504c0dbb09389db6...,paulikallio,da424895-3680-4cf4-83a3-3183eba72227,5794,ABOA,2025,Primary PI,other,"supported team's wellbeing, provided reagents ..."
19,64ce6ddf580275e51b73def9ae9a8d9d983f887a1bf212...,kalisa,4c4d9945-3245-4c0c-ad59-9c8a9aa39add,5794,ABOA,2025,Student Leader,other,NaN
36,7051093d1775f58eac4dde846c41b5e52ead170070126b...,msgera,a0a4993c-0a7d-469b-859d-d4253672d631,5794,ABOA,2025,Student,other,NaN
57,5e2b9973816ffd221575f7b051ef830f280bf491e4fbc4...,riinamatas,d67bddc5-ef7d-4b13-aa03-815dc4af6e49,5794,ABOA,2025,Student Leader,other,Participated in planning of the educational ma...
71,c8b95c441f70cda106d5f12d91d47f2d2304f0bb431a4b...,tyttisandholm,ab2b30bc-1c57-4c4a-9d12-b249defc6cad,5794,ABOA,2025,Student,other,NaN
...,...,...,...,...,...,...,...,...,...
51627,9391f20025cf10b0f53fe63d0ba7942d2104804da7067d...,charline,061957f3-2338-40fd-966a-de0f78d03ccb,5822,ZZU-China,2025,Student,other,Molecular Cloning: Performed vector constructi...
51628,d99d86d3639390e0c699b68b6e8cc5fb79b2b606fa1e08...,sheepsheep,0d711984-6fba-423f-a52f-9b28145e39b0,5822,ZZU-China,2025,Student,other,Human Practice: Focused on cooperation and inc...
51635,5c685a0e1e839152bd5eac105d0d20725ace9be270bb83...,sylus,363f26f7-ed9a-4d14-8455-b9bc8fa25c1f,5822,ZZU-China,2025,Student,other,NaN
51639,9755911438fd73f26e8a81da6575424d3448465cdaaa95...,samdong2017,70dec7cd-52f2-4c1c-a910-56462a60ed82,5822,ZZU-China,2025,Other,other,SanDong is a mentor from China with a strong b...


In [97]:
attributions_2025["Task"].value_counts()

Task
writing                   8620
conceptualization         5259
background research       5152
investigation             4889
public engagement         4318
analysis                  4293
visualization             3740
project administration    3184
data curation             2357
software                  2351
safety                    2184
other                     2169
fundraising               1636
entrepreneurship           986
hardware                   503
validation                   1
supervision                  1
drylab modeling              1
Name: count, dtype: int64

In [99]:
# Check if there are survey participant names in the attribution data that have some of the "missing tasks", print such rows


# Filter attributions_2025 for rows where: the person's hash is in the survey participants list and their task is one of the missing ones
survey_participant_missing_tasks = attributions_2025_clean.loc[
    attributions_2025_clean["FullName"].isin(tasks_participant_names)
    & attributions_2025_clean["Task"].isin(missing_tasks)
]

# print("Survey participants in attributions data with missing tasks:")
# print(survey_participant_missing_tasks[["FullName", "Task", "Team", "Role"]])

if not survey_participant_missing_tasks.empty:
    print("\nSummary by participant:")
    print(
        survey_participant_missing_tasks.groupby("FullName")["Task"]
        .unique()
        .apply(lambda x: ", ".join(x))
    )
else:
    print("No survey participants have missing tasks in attributions data.")



Summary by participant:
FullName
248470f6e20b6bd17be5b18be69c353df79a9d833b2b05cf9d6a5e8253f3957b    other
44e0b4257b5d6473970e6332b76b5c0005e21b5d22a749be346729bbec4d17ed    other
4e2f5187e6bfc2119e0dea8aa8705ce7e5c41f8fddc340bcbfdebc0fbc5d691a    other
5520135e1e537bc0bdedfd6fcadaae62196285e94cd451eb1564b35f5c33e376    other
78266e96943cdd7075a3fcda2c87410cd5cdd91f98fc8f83e7a1cbc977a3231b    other
9b78bc7088d9dd24a005a6e5d966d8445079e74d7ce6f7880b0e7aba2cf6da0e    other
abcdf154b6aaeabd3361ad73c5a1d6dcd1e549f476520e12050150da99b58dd0    other
b22c9c4ec14a48ddac5e0deab3e1366401fa852831436a0c581f8ceea7597926    other
b79d11121382324104021fd4e2b11f68b7fddc5e54f9356939e78d02e1ed8cf5    other
bb17916059aa66071b6d2411bfb59a4bfd2d7df6a1916e3de8d193b1841c9ad4    other
d76fbe3055dfc22aba0af8726e86c69f87be66df815f9a5268414053e9957f06    other
f2f9b9651d5507537e5229d8496b4cc8e4eca396b3edd461aa79acff5e772bc4    other
Name: Task, dtype: object
